# CalculusSolver — Google Colab Training Notebook

This notebook trains and benchmarks the `CalculusSolverModel` using GPU acceleration.

### Features included:
1. **GPU hardware verification** (NVIDIA T4 / V100 / A100)
2. **Repository setup & dependency installation**
3. **Dataset generation & contrast pair generation**
4. **Verified training execution** (15 epochs, cosine LR schedule, 1,500 warmup steps, token-weighted loss)
5. **Controlled experiment suite** (`baseline`, `weighted_loss`, `contrast`, `combined`)
6. **Pure neural benchmark evaluation** (`--no-fallback`)
7. **Saving checkpoints to Google Drive**

## Step 1: Verify GPU Acceleration
Make sure your Colab runtime is set to GPU: **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU (or A100)**.

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Switch runtime type to GPU for fast training!")

## Step 2: Clone Repository

In [ ]:
%cd /content
!rm -rf /content/CalculusSolver
!git clone https://github.com/umer12x/CalculusSolver.git
%cd /content/CalculusSolver

## Step 3: Install Dependencies

In [ ]:
!pip install -q -r requirements-neural.txt
!pip install -q pytest pytest-subtests

## Step 4: Prepare Datasets & Contrast Pairs
Verify that the dataset splits and contrast pairs exist; generate them if missing.

In [ ]:
import os

# Generate standard training data if splits don't exist
if not os.path.exists("data/splits/train.jsonl"):
    print("Generating dataset splits...")
    !python problem_generator.py
else:
    print("Standard dataset splits found.")

# Generate contrast pairs for variable-binding minibatches
if not os.path.exists("data/splits/contrast_train.jsonl"):
    print("Generating contrast dataset...")
    !python problem_generator.py --contrast-only
else:
    print("Contrast dataset splits found.")

## Step 5: Run Preflight Unit Tests
Quickly check that loss weighting, contrast batching, schedule, and model contracts are healthy.

In [ ]:
!python -m pytest tests/unit/test_token_weighted_loss.py tests/unit/test_contrast_pairs.py tests/unit/test_training_schedule.py tests/unit/test_training_reproducibility.py -q

## Step 6: Train the Model

### Choice 1: Standard Full Training Run (15 Epochs)
This runs the default 15-epoch training with cosine decay, 1500 warmup steps, token-weighted loss, and contrast batches.

In [ ]:
!python train.py

### Choice 2: Controlled Experiments (Baseline vs Weighted Loss vs Contrast vs Combined)
If you want to run the 4 ablation arms and compare them with strict factor isolation:

In [ ]:
# Run all 4 arms:
# !python scripts/run_experiments.py

# Or run a specific arm (e.g. combined only):
# !python scripts/run_experiments.py --arms combined

## Step 7: Evaluate Model Performance
Run evaluation against the official benchmark suite.
Note: pure-neural diagnostic runs require `--output` and `--json-output` flags so they do not overwrite canonical reports.

In [ ]:
# Option A: Pure neural evaluation (without verifier fallback)
!python eval/run_eval.py --no-fallback --output docs/EVAL_RESULTS_NEURAL.md --json-output docs/eval_results_neural.json

# Option B: Full canonical evaluation (hybrid solver with verifier fallback)
# !python eval/run_eval.py

## Step 8: Publish Best Checkpoint
Selects the top-performing arm based on validation exact match and pure-neural benchmark score, updates reports, and prepares the final bundle.

In [ ]:
# Show selection ranking without altering files
!python scripts/publish_best_checkpoint.py --dry-run

# When ready, publish the chosen checkpoint and refresh canonical evaluation
# !python scripts/publish_best_checkpoint.py --yes --refresh-canonical-eval

## Step 9: Save Checkpoints to Google Drive
Mount your Google Drive to permanently preserve your trained model weights and evaluation reports.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

# Create target folder in Google Drive
drive_target = "/content/drive/MyDrive/CalculusSolver_Checkpoints"
os.makedirs(drive_target, exist_ok=True)

# Copy final checkpoint and reports
!cp -v checkpoints/final/best.pt /content/drive/MyDrive/CalculusSolver_Checkpoints/
!cp -v docs/TRAINING_RESULTS.md /content/drive/MyDrive/CalculusSolver_Checkpoints/
!cp -v docs/eval_results.json /content/drive/MyDrive/CalculusSolver_Checkpoints/
!cp -v docs/EVAL_RESULTS.md /content/drive/MyDrive/CalculusSolver_Checkpoints/

print("Artifacts successfully backed up to Google Drive!")